In [59]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
# import xgboost as xgb
from scipy import interpolate
from scipy.stats import linregress
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from pprint import pprint

In [45]:
# ------------------------------
# Configuration
# ------------------------------
DATA_ROOT = Path("../../Data")            # Folder containing S1, S2, ...
GRADES_CSV = Path("../../outputs/grades_wide.csv")       # CSV with columns: student_id,final,midterm_1,midterm_2
TARGET_FS = 4.0                 # Resample all signals to 4 Hz
WINDOW_SEC = 60                 # 60 seconds window
STEP_SEC = 30                   # 30 seconds stride
BINARY_THRESHOLD = 70.0         # Grade >= 70 -> low stress (0), else high stress (1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3

In [46]:
# ------------------------------
# 1. Load grades
# ------------------------------
def load_grades(csv_path):
    df = pd.read_csv(csv_path)
    # Convert student_id to standard format (e.g., S01)
    df['student_id'] = df['student_id'].str.upper()
    # Final is out of 200, convert to percent
    df['final'] = df['final'] / 2.0
    # Rename columns to match exam folder names
    grades_dict = {
        'Midterm 1': dict(zip(df['student_id'], df['midterm_1'])),
        'Midterm 2': dict(zip(df['student_id'], df['midterm_2'])),
        'Final': dict(zip(df['student_id'], df['final']))
    }
    return grades_dict

grades_dict = load_grades(GRADES_CSV)


In [52]:
# ------------------------------
# 2. Load multimodal signals from a CSV file (dataset format)
# ------------------------------
def load_signal(csv_path, cols=None):
    """
    Load CSV with format:
    row0: start timestamp
    row1: sampling rate (Hz)
    rows2..end: data samples
    Returns (timestamps, data_array, fs)
    """
    df = pd.read_csv(csv_path, header=None)
    start_time = df.iloc[0, 0]
    fs = df.iloc[1, 0]
    data = df.iloc[2:].values.astype(float)
    if cols is not None:
        data = data[:, cols]
    if data.ndim == 1:
        data = data[:, None]
    n_samples = data.shape[0]
    timestamps = start_time + np.arange(n_samples) / fs
    return timestamps, data, fs


In [53]:
def resample_signal(timestamps, data, orig_fs, target_fs=TARGET_FS):
    """Resample multivariate signal to target_fs using linear interpolation."""
    t_start, t_end = timestamps[0], timestamps[-1]
    new_t = np.arange(t_start, t_end, 1.0/target_fs)
    resampled = np.zeros((len(new_t), data.shape[1]))
    for ch in range(data.shape[1]):
        f = interpolate.interp1d(timestamps, data[:, ch], kind='linear', fill_value='extrapolate')
        resampled[:, ch] = f(new_t)
    return new_t, resampled

In [54]:
def load_session(subj_folder, exam_name):
    """Load all modalities for one exam session."""
    exam_path = os.path.join(subj_folder, exam_name)
    if not os.path.exists(exam_path):
        print(f"  Warning: {exam_path} not found")
        return None
    
    signals = {}
    
    # Load regular modalities
    mods = {
        'ACC': ('ACC.csv', [0,1,2]),  # 3 axes
        'BVP': ('BVP.csv', 0),
        'EDA': ('EDA.csv', 0),
        'HR': ('HR.csv', 0),
        'TEMP': ('TEMP.csv', 0)
    }
    
    for name, (fname, cols) in mods.items():
        path = os.path.join(exam_path, fname)
        if not os.path.exists(path):
            print(f"  Warning: {path} not found")
            continue
        try:
            ts, data, fs = load_signal(path, cols)
            if data.size == 0:
                continue
            new_ts, resampled = resample_signal(ts, data, fs, TARGET_FS)
            signals[name] = (new_ts, resampled)
        except Exception as e:
            print(f"  Error loading {name}: {e}")
            continue
    
    # Load IBI for HRV features
    ibi_path = os.path.join(exam_path, "IBI.csv")
    if os.path.exists(ibi_path):
        try:
            df_ibi = pd.read_csv(ibi_path, header=None)
            start_time = df_ibi.iloc[0, 0]
            ibi_vals = df_ibi.iloc[2:, 0].values.astype(float)
            ibi_times = start_time + np.cumsum(ibi_vals)
            signals['IBI_times'] = ibi_times
            signals['IBI_vals'] = ibi_vals
        except Exception as e:
            print(f"  Error loading IBI: {e}")
            signals['IBI_times'] = np.array([])
            signals['IBI_vals'] = np.array([])
    else:
        signals['IBI_times'] = np.array([])
        signals['IBI_vals'] = np.array([])
    
    # Get grade for this exam
    subj_name = os.path.basename(subj_folder)
    # Normalize subject name (S01 -> S1, S1 -> S1)
    match = re.search(r'S(\d+)', subj_name.upper())
    if match:
        subj_id = f"S{int(match.group(1))}"
    else:
        subj_id = subj_name.upper()
    
    grade = grades_dict[exam_name].get(subj_id, None)
    if grade is None:
        print(f"  Warning: No grade found for {subj_id} - {exam_name}")
        return None
    
    signals['grade'] = grade
    signals['binary_label'] = 1.0 if grade < BINARY_THRESHOLD else 0.0
    signals['subject_id'] = subj_id
    
    return signals

In [55]:
# ------------------------------
# 3. Window extraction & feature engineering
# ------------------------------
def compute_hrv_features(ibi_times, ibi_vals, window_start, window_end):
    """Compute HRV features from IBI events within time window."""
    mask = (ibi_times >= window_start) & (ibi_times <= window_end)
    ibi_in_win = ibi_vals[mask]
    if len(ibi_in_win) < 2:
        return [0.0, 0.0, 0.0]
    
    diff = np.diff(ibi_in_win)
    rmssd = np.sqrt(np.mean(diff**2))
    sdnn = np.std(ibi_in_win)
    mean_hr = 60.0 / np.mean(ibi_in_win)
    return [rmssd, sdnn, mean_hr]

def extract_windows(signals, window_sec, step_sec, fs=TARGET_FS):
    """
    Extract windows from a session.
    Returns (static_features_list, sequences_list, labels_list)
    """
    if 'ACC' not in signals:
        return [], [], []
    
    t_acc, acc_data = signals['ACC']
    t_start, t_end = t_acc[0], t_acc[-1]
    win_len = int(window_sec * fs)
    step_len = int(step_sec * fs)
    
    # Combine all resampled signals
    mod_list = ['ACC', 'BVP', 'EDA', 'HR', 'TEMP']
    raw_data_list = []
    for mod in mod_list:
        if mod in signals:
            _, data = signals[mod]
            raw_data_list.append(data)
    
    if not raw_data_list:
        return [], [], []
    
    # Truncate to same length
    min_len = min(d.shape[0] for d in raw_data_list)
    raw_data = np.concatenate([d[:min_len] for d in raw_data_list], axis=1)
    t_axis = t_acc[:min_len]
    
    # IBI data
    ibi_times = signals.get('IBI_times', np.array([]))
    ibi_vals = signals.get('IBI_vals', np.array([]))
    
    static_feats = []
    sequences = []
    labels = []
    
    for start_idx in range(0, min_len - win_len + 1, step_len):
        end_idx = start_idx + win_len
        win_t_start = t_axis[start_idx]
        win_t_end = t_axis[end_idx-1]
        X_raw = raw_data[start_idx:end_idx, :]
        
        # Static features: mean, std, min, max, slope per channel
        static = []
        for ch in range(X_raw.shape[1]):
            mean_val = np.mean(X_raw[:, ch])
            std_val = np.std(X_raw[:, ch])
            min_val = np.min(X_raw[:, ch])
            max_val = np.max(X_raw[:, ch])
            # Linear trend (slope)
            x_vals = np.arange(X_raw.shape[0])
            slope, _, _, _, _ = linregress(x_vals, X_raw[:, ch])
            static.extend([mean_val, std_val, min_val, max_val, slope])
        
        # HRV features
        hrv = compute_hrv_features(ibi_times, ibi_vals, win_t_start, win_t_end)
        static.extend(hrv)
        
        static_feats.append(np.array(static, dtype=np.float32))
        sequences.append(X_raw.astype(np.float32))
        labels.append(signals['binary_label'])
    
    return static_feats, sequences, labels



In [56]:
# ------------------------------
# 4. Load all data across participants and exams
# ------------------------------
participants = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.upper().startswith('S')]
participants.sort()
print(f"\nFound participant folders: {participants}")

all_samples = []  # Each: {'static': array, 'seq': array, 'label': int, 'subject': str}

for subj_folder in participants:
    subj_path = os.path.join(DATA_ROOT, subj_folder)
    print(f"\nProcessing {subj_folder}...")
    
    for exam in ['Midterm 1', 'Midterm 2', 'Final']:
        print(f"  Loading {exam}...")
        signals = load_session(subj_path, exam)
        if signals is None:
            print(f"    Skipping {exam} (no data or grade)")
            continue
        
        static_list, seq_list, label_list = extract_windows(signals, WINDOW_SEC, STEP_SEC, TARGET_FS)
        
        if len(static_list) == 0:
            print(f"    No windows extracted from {exam}")
            continue
        
        print(f"    Extracted {len(static_list)} windows")
        
        for s, seq, lab in zip(static_list, seq_list, label_list):
            all_samples.append({
                'static': s,
                'seq': seq,
                'label': lab,
                'subject': signals['subject_id']  # Use normalized subject ID
            })

print(f"\nTotal samples (windows) extracted: {len(all_samples)}")

# Display subject distribution
subject_counts = {}
for sample in all_samples:
    subj = sample['subject']
    subject_counts[subj] = subject_counts.get(subj, 0) + 1
print("\nSamples per subject:")
for subj, count in sorted(subject_counts.items()):
    print(f"  {subj}: {count} windows")



Found participant folders: ['S1', 'S10', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Processing S1...
  Loading Midterm 1...
    Extracted 371 windows
  Loading Midterm 2...
    Extracted 369 windows
  Loading Final...
    Extracted 778 windows

Processing S10...
  Loading Midterm 1...
    Extracted 388 windows
  Loading Midterm 2...
    Extracted 431 windows
  Loading Final...
    Extracted 767 windows

Processing S2...
  Loading Midterm 1...
    Extracted 398 windows
  Loading Midterm 2...
    Extracted 461 windows
  Loading Final...
    Extracted 843 windows

Processing S3...
  Loading Midterm 1...
    Extracted 405 windows
  Loading Midterm 2...
    Extracted 340 windows
  Loading Final...
    Extracted 859 windows

Processing S4...
  Loading Midterm 1...
    Extracted 388 windows
  Loading Midterm 2...
    Extracted 445 windows
  Loading Final...
    Extracted 529 windows

Processing S5...
  Loading Midterm 1...
    Extracted 398 windows
  Loading Midterm 2...
    Extracted 

In [57]:
# ------------------------------
# 5. Split data by subject (no mixing)
# ------------------------------
# Define splits using the same normalized format (S1, S2, ...)
train_subjects = [f"S{i}" for i in range(1, 8)]    # S1, S2, S3, S4, S5, S6, S7
val_subjects   = [f"S{i}" for i in range(8, 9)]    # S8
test_subjects  = [f"S{i}" for i in range(9, 11)]   # S9, S10

print(f"\nSplit definition:")
print(f"  Train subjects: {train_subjects}")
print(f"  Validation subjects: {val_subjects}")
print(f"  Test subjects: {test_subjects}")

# Assign samples to splits
train_samples = [s for s in all_samples if s['subject'] in train_subjects]
val_samples   = [s for s in all_samples if s['subject'] in val_subjects]
test_samples  = [s for s in all_samples if s['subject'] in test_subjects]

print(f"\nSplit results:")
print(f"  Train samples: {len(train_samples)}")
print(f"  Validation samples: {len(val_samples)}")
print(f"  Test samples: {len(test_samples)}")

# Check for unassigned subjects
assigned_subjects = set(train_subjects + val_subjects + test_subjects)
all_subjects_in_data = set(s['subject'] for s in all_samples)
unassigned_subjects = all_subjects_in_data - assigned_subjects
if unassigned_subjects:
    print(f"\n⚠️  Warning: These subjects are in data but not assigned to any split: {unassigned_subjects}")
    print("They will be ignored. Add them to train/val/test if needed.")

if len(train_samples) == 0 or len(val_samples) == 0 or len(test_samples) == 0:
    print("\n❌ ERROR: One or more splits are empty!")
    print("Please check that your folder names match the split lists.")
    print(f"Expected subjects: {assigned_subjects}")
    print(f"Found subjects in data: {all_subjects_in_data}")
    exit(1)

# Extract static features and labels for traditional ML
X_train_static = np.array([s['static'] for s in train_samples])
y_train = np.array([s['label'] for s in train_samples])
X_val_static = np.array([s['static'] for s in val_samples])
y_val = np.array([s['label'] for s in val_samples])
X_test_static = np.array([s['static'] for s in test_samples])
y_test = np.array([s['label'] for s in test_samples])

# Standardize static features
scaler = StandardScaler()
X_train_static = scaler.fit_transform(X_train_static)
X_val_static = scaler.transform(X_val_static)
X_test_static = scaler.transform(X_test_static)

print(f"\nStatic feature shape: {X_train_static.shape[1]} features per window")

# Extract sequences for deep learning
train_seqs = [s['seq'] for s in train_samples]
train_labels = [s['label'] for s in train_samples]
val_seqs = [s['seq'] for s in val_samples]
val_labels = [s['label'] for s in val_samples]
test_seqs = [s['seq'] for s in test_samples]
test_labels = [s['label'] for s in test_samples]

print(f"Sequence shape: {train_seqs[0].shape if train_seqs else 'None'}")

# Check class balance
def print_class_balance(y, name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name} class balance:")
    for cls, cnt in zip(unique, counts):
        stress_level = "High Stress" if cls == 1 else "Low Stress"
        print(f"  {stress_level} (class {int(cls)}): {cnt} ({cnt/len(y)*100:.1f}%)")

print("\nClass balance:")
print_class_balance(y_train, "Train")
print_class_balance(y_val, "Validation")
print_class_balance(y_test, "Test")


Split definition:
  Train subjects: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']
  Validation subjects: ['S8']
  Test subjects: ['S9', 'S10']

Split results:
  Train samples: 10549
  Validation samples: 1283
  Test samples: 2892

Static feature shape: 38 features per window
Sequence shape: (240, 7)

Class balance:
Train class balance:
  Low Stress (class 0): 8257 (78.3%)
  High Stress (class 1): 2292 (21.7%)
Validation class balance:
  Low Stress (class 0): 1283 (100.0%)
Test class balance:
  Low Stress (class 0): 809 (28.0%)
  High Stress (class 1): 2083 (72.0%)


In [60]:
# ------------------------------
# 6. Traditional ML Models
# ------------------------------
print("\n" + "="*60)
print("TRAINING TRADITIONAL ML MODELS")
print("="*60)

def train_eval_ml(model, model_name, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, test_pred)
    
    print(f"\n{model_name}:")
    print(f"  Validation Accuracy: {val_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  Test F1 Score: {test_f1:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    
    return {'acc': test_acc, 'f1': test_f1, 'auc': test_auc}

ml_models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    # 'XGBoost': xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

ml_results = {}
for name, model in ml_models.items():
    print(f"\nTraining {name}...")
    res = train_eval_ml(model, name, X_train_static, y_train, X_val_static, y_val, X_test_static, y_test)
    ml_results[name] = res



TRAINING TRADITIONAL ML MODELS

Training Decision Tree...

Decision Tree:
  Validation Accuracy: 0.8792
  Test Accuracy: 0.3683
  Test F1 Score: 0.2202
  Test AUC: 0.5607

Training SVM (RBF)...

SVM (RBF):
  Validation Accuracy: 0.9119
  Test Accuracy: 0.2891
  Test F1 Score: 0.0437
  Test AUC: 0.4989

Training Random Forest...

Random Forest:
  Validation Accuracy: 0.8870
  Test Accuracy: 0.3607
  Test F1 Score: 0.2040
  Test AUC: 0.5550


In [61]:
# ------------------------------
# 7. Deep Learning Models (PyTorch)
# ------------------------------
print("\n" + "="*60)
print("TRAINING DEEP LEARNING MODELS")
print("="*60)

class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        X = torch.tensor(self.sequences[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return X, y

train_dataset = SequenceDataset(train_seqs, train_labels)
val_dataset = SequenceDataset(val_seqs, val_labels)
test_dataset = SequenceDataset(test_seqs, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Model definitions
class LSTMWithoutAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = h_n[-1]
        return self.classifier(out).squeeze()

class GRUWithoutAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, h_n = self.gru(x)
        out = h_n[-1]
        return self.classifier(out).squeeze()

class Attention(nn.Module):
    """Additive attention mechanism"""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        score = self.v(torch.tanh(self.W(lstm_out)))
        attn_weights = torch.softmax(score, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class LSTMWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.attention = Attention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, attn_weights = self.attention(lstm_out)
        return self.classifier(context).squeeze(), attn_weights

class GRUWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.attention = Attention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        gru_out, _ = self.gru(x)
        context, attn_weights = self.attention(gru_out)
        return self.classifier(context).squeeze(), attn_weights



TRAINING DEEP LEARNING MODELS


In [62]:
def train_dl_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)
    
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            
            if hasattr(model, 'attention'):
                output, _ = model(X)
            else:
                output = model(X)
            
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                if hasattr(model, 'attention'):
                    output, _ = model(X)
                else:
                    output = model(X)
                loss = criterion(output, y)
                val_loss += loss.item()
                probs = torch.sigmoid(output)
                preds = (probs > 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(all_labels, all_preds)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # Load best model
    model.load_state_dict(best_model_state)
    return model

In [63]:
def evaluate_dl_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_attn_weights = []
    
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if hasattr(model, 'attention'):
                output, attn = model(X)
                all_attn_weights.append(attn.cpu().numpy())
            else:
                output = model(X)
            
            probs = torch.sigmoid(output)
            preds = (probs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_preds)
    
    print(f"\n{model.__class__.__name__} Results:")
    print(f"  Test Accuracy: {acc:.4f}")
    print(f"  Test F1 Score: {f1:.4f}")
    print(f"  Test AUC: {auc:.4f}")
    
    return {'acc': acc, 'f1': f1, 'auc': auc}, all_attn_weights


In [64]:
# Get input dimension
input_dim = train_seqs[0].shape[1]
print(f"\nInput feature dimension: {input_dim}")

dl_models_dict = {
    'LSTM': LSTMWithoutAttention(input_dim),
    'GRU': GRUWithoutAttention(input_dim),
    'LSTM+Attention': LSTMWithAttention(input_dim),
    'GRU+Attention': GRUWithAttention(input_dim)
}

dl_results = {}
attention_weights = {}

for name, model in dl_models_dict.items():
    print(f"\n{'='*40}")
    print(f"Training {name}")
    print('='*40)
    trained_model = train_dl_model(model, train_loader, val_loader)
    metrics, attn = evaluate_dl_model(trained_model, test_loader)
    dl_results[name] = metrics
    if 'Attention' in name and attn:
        attention_weights[name] = attn



Input feature dimension: 7

Training LSTM
Epoch 10/50 - Train Loss: 0.3125, Val Loss: 0.2837, Val Acc: 0.8745
Epoch 20/50 - Train Loss: 0.2659, Val Loss: 0.3744, Val Acc: 0.8465
Epoch 30/50 - Train Loss: 0.2614, Val Loss: 0.3808, Val Acc: 0.8426
Epoch 40/50 - Train Loss: 0.2606, Val Loss: 0.3809, Val Acc: 0.8426
Epoch 50/50 - Train Loss: 0.2613, Val Loss: 0.3809, Val Acc: 0.8426

LSTMWithoutAttention Results:
  Test Accuracy: 0.3005
  Test F1 Score: 0.0724
  Test AUC: 0.5072

Training GRU
Epoch 10/50 - Train Loss: 0.1593, Val Loss: 0.7749, Val Acc: 0.7747
Epoch 20/50 - Train Loss: 0.0944, Val Loss: 1.1483, Val Acc: 0.7155
Epoch 30/50 - Train Loss: 0.0916, Val Loss: 1.1432, Val Acc: 0.7217
Epoch 40/50 - Train Loss: 0.0933, Val Loss: 1.1434, Val Acc: 0.7217
Epoch 50/50 - Train Loss: 0.0923, Val Loss: 1.1435, Val Acc: 0.7217

GRUWithoutAttention Results:
  Test Accuracy: 0.3033
  Test F1 Score: 0.0911
  Test AUC: 0.5038

Training LSTM+Attention
Epoch 10/50 - Train Loss: 0.2301, Val Loss:

In [ ]:
# ------------------------------
# 8. Results Comparison
# ------------------------------
print("\n" + "="*60)
print("FINAL RESULTS COMPARISON")
print("="*60)

# Combine all results
all_results = {**ml_results, **dl_results}

# Create comparison table
results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values('acc', ascending=False)
print("\nModel Performance (sorted by Accuracy):")
print(results_df.to_string())

# Plotting
fig, axes = plt.subplots(1, 3, figsize=(15, 8))

# Accuracy comparison
models = results_df.index.tolist()
acc_scores = results_df['acc'].tolist()
f1_scores = results_df['f1'].tolist()
auc_scores = results_df['auc'].tolist()

# Horizontal bar chart for better readability
y_pos = np.arange(len(models))

axes[0].barh(y_pos, acc_scores, color='skyblue')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(models)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)

axes[1].barh(y_pos, f1_scores, color='lightgreen')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(models)
axes[1].set_xlabel('F1 Score')
axes[1].set_title('F1 Score Comparison')
axes[1].set_xlim(0, 1)

axes[2].barh(y_pos, auc_scores, color='salmon')
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(models)
axes[2].set_xlabel('AUC')
axes[2].set_title('AUC-ROC Comparison')
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ------------------------------
# 9. Attention Visualization
# ------------------------------
def plot_attention_weights(attn_weights_list, model_name, num_samples=3):
    """Plot attention weights for a few test samples"""
    if not attn_weights_list:
        print(f"No attention weights to plot for {model_name}")
        return
    
    # Concatenate attention weights from all batches
    all_attn = np.concatenate(attn_weights_list, axis=0)  # (num_samples, seq_len, 1)
    all_attn = all_attn.squeeze(-1)  # (num_samples, seq_len)
    
    # Plot first few samples
    n_plot = min(num_samples, all_attn.shape[0])
    fig, axes = plt.subplots(n_plot, 1, figsize=(12, 3*n_plot))
    if n_plot == 1:
        axes = [axes]
    
    for i in range(n_plot):
        axes[i].plot(all_attn[i], 'b-', linewidth=2)
        axes[i].fill_between(range(len(all_attn[i])), all_attn[i], alpha=0.3)
        axes[i].set_ylabel('Attention Weight')
        axes[i].set_xlabel('Time Step (60-second window)')
        axes[i].set_title(f'{model_name} - Sample {i+1}')
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle(f'Attention Weight Distribution - {model_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{model_name}_attention_weights.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print(f"\n{model_name} Attention Statistics:")
    print(f"  Mean attention weight: {np.mean(all_attn):.4f}")
    print(f"  Std attention weight: {np.std(all_attn):.4f}")
    print(f"  Max attention weight: {np.max(all_attn):.4f}")
    print(f"  Peak attention at time step: {np.argmax(all_attn, axis=1).mean():.1f}")

# Plot attention for models that have it
for model_name, attn_weights_list in attention_weights.items():
    if attn_weights_list:
        plot_attention_weights(attn_weights_list, model_name)

print("\n✅ All experiments completed successfully!")
print(f"Results saved to: model_comparison.png and individual attention plots")
